# 07 - Final forecast and submission

The selected approach is a 55% seasonal-naive / 45% global Poisson HGB blend. The global weight is tuned on one temporal origin and evaluated on the other; the leave-one-origin-out results are written to `outputs/model_selection/leave_one_origin_out_blend_validation.csv`. Forecasts enforce launch/EOL bounds, and a launched replacement SKU with no own history uses its predecessor as an explicitly documented 1.0 transfer assumption.

Predictions are recursive for horizons beyond 12 months. The submission is rounded to nonnegative integers and checked against the exact test keys and required columns. For a full reproducible run, use `python -m src.pipeline` from the project root.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "outputs" / "features").is_dir():
            return candidate
    raise FileNotFoundError("Run notebooks 01-06 before producing the submission.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src.forecasting import create_final_submission

summary = create_final_submission(PROJECT_ROOT)
summary

In [ ]:
import pandas as pd
from IPython.display import display

submission_path = PROJECT_ROOT / "outputs" / "submission" / "final_submission.csv"
audit_path = PROJECT_ROOT / "outputs" / "submission" / "forecast_audit.csv"
submission = pd.read_csv(submission_path, low_memory=False)
audit = pd.read_csv(audit_path, low_memory=False)
test = pd.read_csv(PROJECT_ROOT / "outputs" / "cleaned" / "test_cleaned.csv", low_memory=False)

display(submission.head())
display(audit["Forecast_Method"].value_counts().rename_axis("Method").to_frame("Rows"))

required_columns = ["Date", "SKU_ID", "Region", "Units_Sold_Forecast"]
assert list(submission.columns) == required_columns
assert len(submission) == len(test) == 58_219
assert not submission.duplicated(["Date", "SKU_ID", "Region"]).any()
assert submission[required_columns].notna().all().all()
assert submission["Units_Sold_Forecast"].ge(0).all()
assert pd.api.types.is_integer_dtype(submission["Units_Sold_Forecast"])
assert submission[["Date", "SKU_ID", "Region"]].sort_values(
    ["Date", "SKU_ID", "Region"]
).reset_index(drop=True).equals(
    test[["Date", "SKU_ID", "Region"]].sort_values(
        ["Date", "SKU_ID", "Region"]
    ).reset_index(drop=True)
)
print(f"Validated submission: {len(submission):,} rows, exact schema, complete unique keys, nonnegative integer forecasts.")

Deliver `outputs/submission/final_submission.csv`. Keep `forecast_audit.csv` with the project so every forecast can be traced to its seasonal, global, replacement, or fallback source.